## 1.数据预处理

### 1.1 数据读取

In [ ]:
with open('data/FinancialPhraseBank/Sentences_75Agree.txt', 'r') as f: 
    FPB_75_in = f.read() #Malo数据特色，分为不同置信度的样本
with open('data/FinancialPhraseBank/Sentences_AllAgree.txt', 'r') as f:
    FPB_100_in = f.read()

In [ ]:
FPB_75_in[:200]

In [ ]:
from string import punctuation

FPB_75_lines = FPB_75_in.split('\n')

FPB_75_data = []
FPB_75_label = []
tags = {'negative':0,'neutral':1,'positive':2}
for line in FPB_75_lines[:-1]:
    #line = line.lower()  不用统一大小写
    #data_line = line.split('@')[0]
    FPB_75_label.append(tags[line.split('@')[1]])
    #data_line = ''.join([c for c in data_line if c not in punctuation]) #不用去除标点
    #data_l = data_line.split(' ') 原始句子而非 list of words
    #data_l = [c for c in data_l if len(c)>0] #Malo这份数据特别之处，标点前有空格，故而去除标点后再以空格区分，出现了一些空值
    FPB_75_data.append(line.split('@')[0])
    

In [ ]:
print(FPB_75_data[0],'\n',FPB_75_label[0])

In [ ]:
FPB_100_lines = FPB_100_in.split('\n')
FPB_100_data = []
FPB_100_label = []
tags = {'negative':0,'neutral':1,'positive':2}
for line in FPB_100_lines[:-1]:
    FPB_75_label.append(tags[line.split('@')[1]])
    FPB_75_data.append(line.split('@')[0])

In [ ]:
FPB_data_raw = FPB_75_data + FPB_100_data
FPB_label_raw = FPB_75_label + FPB_100_label

In [ ]:
import json

# 保存到 JSON 文件
with open("data/FinancialPhraseBank/labels.json", "w", encoding="utf-8") as json_file:
    json.dump(FPB_label_raw, json_file, ensure_ascii=False, indent=4)

In [ ]:
with open("data/FinancialPhraseBank/eng_sent.txt", "w", encoding="utf-8") as file:
    for sentence in FPB_data_raw:
        file.write(sentence + "\n")

### 1.2 Tokenizer

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW  # 改为从PyTorch导入AdamW
from transformers import get_scheduler
import evaluate
metric = evaluate.load("accuracy")

In [ ]:
!pip install scikit-learn

In [ ]:
#需要请夏威夷的朋友帮忙下载模型
# 下载 bert-base tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
tokenizer.save_pretrained("./model/bert-base-uncased-tokenizer")
#下载bert-base 模型
model = BertForSequenceClassification.from_pretrained("bert-base-uncased")
model.save_pretrained("./model/bert-base-uncased-model")



In [ ]:
# 加载BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("./model/bert-base-uncased-tokenizer")

In [ ]:
# 使用 BERT 的 tokenizer 对句子进行分词和编码
encodings = tokenizer(
    FPB_data_raw,                  # 输入的句子列表
    truncation=True,            # 是否截断超出最大长度的句子
    padding=True,               # 是否对较短的句子进行填充，使其达到统一长度
    max_length=128,             # 最大句子长度，超过的部分会被截断
    return_tensors="pt"         # 返回 PyTorch 格式的张量
)

# 获取编码后的 input_ids 和 attention_mask
input_ids = encodings["input_ids"]               # 编码后的 token 序列，类似于word2vec时的int id
attention_mask = encodings["attention_mask"]     # 注意力掩码，用于区分实际内容和填充部分，即区分真实id和padding
labels = torch.tensor(FPB_label_raw)                    # 将标签转为 PyTorch 张量

# 创建数据集和数据加载器
# 使用 PyTorch 的 TensorDataset 封装数据，包括 input_ids, attention_mask 和 labels
dataset = torch.utils.data.TensorDataset(input_ids, attention_mask, labels)

### 1.3 数据集划分

In [ ]:
from torch.utils.data import random_split, DataLoader
# 设置随机种子
torch.manual_seed(42)

# 数据集的划分比例
train_size = int(0.8 * len(dataset))  # 80% 训练集
val_size = int(0.1 * len(dataset))    # 10% 验证集
test_size = len(dataset) - train_size - val_size  # 剩余的 10% 测试集

# 按照比例划分数据集
train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size], generator=torch.Generator().manual_seed(42)
)

# 创建数据加载器
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, drop_last=True)  # 训练数据加载器
val_loader = DataLoader(val_dataset, batch_size=16, drop_last=True)                   # 验证数据加载器
test_loader = DataLoader(test_dataset, batch_size=16, drop_last=True)                 # 测试数据加载器

# 检查划分结果
print(f"Train size: {len(train_dataset)}, Validation size: {len(val_dataset)}, Test size: {len(test_dataset)}")

## 2.BERT分类

### 2.1 线下模型

In [ ]:
# 加载本地预训练的 BERT 模型，忽略大小不匹配的分类头
model_local = BertForSequenceClassification.from_pretrained(
    "./model/bert-base-uncased-model",  # 指定本地模型路径
    num_labels=3,                       # 分类任务的标签数量（这里是三分类）
    ignore_mismatched_sizes=True        # 忽略大小不匹配的问题
)

In [ ]:
optimizer = AdamW(model_local.parameters(), lr=5e-5)
num_training_steps = len(train_loader) * 5  # 假设训练5个epoch
scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)


In [ ]:
import time
device = torch.device("cpu")

In [ ]:
# 加载评估指标
model_local.to(device)
# 训练与验证
epochs = 3
for epoch in range(epochs):
    print(f"Epoch {epoch + 1}/{epochs}")

    # 记录训练开始时间
    start_time = time.time()

    # 训练模型
    model_local.train()  # 设置模型为训练模式
    for batch in train_loader:
        # 将 batch 数据加载到 GPU 或 CPU
        batch = {k: v.to(device) for k, v in zip(["input_ids", "attention_mask", "labels"], batch)}
        
        # 前向传播，计算损失和预测值
        outputs = model_local(**batch)
        loss = outputs.loss          # 计算当前批次的损失
        logits = outputs.logits      # 获取模型的预测结果

        # 反向传播并更新参数
        optimizer.zero_grad()        # 清除优化器的梯度
        loss.backward()              # 计算梯度
        optimizer.step()             # 更新模型参数
        scheduler.step()             # 更新学习率

    # 记录训练结束时间
    end_time = time.time()
    train_time = end_time - start_time  # 计算训练时间
    print(f"Training time: {train_time:.2f} seconds")

    # 验证模型
    model_local.eval()  # 设置模型为评估模式

    # 记录验证开始时间
    start_time = time.time()

    for batch in val_loader:
        batch = {k: v.to(device) for k, v in zip(["input_ids", "attention_mask", "labels"], batch)}
        with torch.no_grad():  # 禁用梯度计算以节省内存和加速
            outputs = model_local(**batch)

        logits = outputs.logits                # 获取预测值
        predictions = torch.argmax(logits, dim=-1)  # 获取预测的类别
        metric.add_batch(predictions=predictions, references=batch["labels"])  # 更新评估指标

    # 记录验证结束时间
    end_time = time.time()
    val_time = end_time - start_time  # 计算验证时间
    print(f"Validation time: {val_time:.2f} seconds")

    # 计算并输出验证集的准确率
    val_accuracy = metric.compute()["accuracy"]
    print(f"Epoch {epoch + 1}: Validation Accuracy = {val_accuracy:.4f}")

In [ ]:
model_local.eval()
for batch in test_loader:
    batch = {k: v.to(device) for k, v in zip(["input_ids", "attention_mask", "labels"], batch)}
    with torch.no_grad():  # 禁用梯度计算以节省内存和加速
        outputs = model_local(**batch)

    logits = outputs.logits                # 获取预测值
    predictions = torch.argmax(logits, dim=-1)  # 获取预测的类别
    metric.add_batch(predictions=predictions, references=batch["labels"])  # 更新评估指标


# 计算并输出验证集的准确率
test_accuracy = metric.compute()["accuracy"]
print(f" Test Accuracy = {test_accuracy:.4f}")

### 2.2 仅微调分类头

In [ ]:
# 加载本地预训练的 BERT 模型，忽略大小不匹配的分类头
model_local_freeze = BertForSequenceClassification.from_pretrained(
    "./model/bert-base-uncased-model",  # 指定本地模型路径
    num_labels=3,                       # 分类任务的标签数量（这里是三分类）
    ignore_mismatched_sizes=True        # 忽略大小不匹配的问题
)

In [ ]:
optimizer = AdamW(model_local_freeze.parameters(), lr=5e-5)
num_training_steps = len(train_loader) * 5  # 假设训练5个epoch
scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)


In [ ]:
import time

model_local_freeze.to(device)
# 冻结 BERT 模型的参数（只训练分类头）
for param in model_local_freeze.bert.parameters():
    param.requires_grad = False

# 加载评估指标
metric = load_metric("accuracy")

# 训练与验证
epochs = 5
for epoch in range(epochs):
    print(f"Epoch {epoch + 1}/{epochs}")

    # 记录训练开始时间
    start_time = time.time()

    # 训练模型
    model_local_freeze.train()  # 设置模型为训练模式
    for batch in train_loader:
        # 将 batch 数据加载到 GPU 或 CPU
        batch = {k: v.to(device) for k, v in zip(["input_ids", "attention_mask", "labels"], batch)}
        
        # 前向传播，计算损失和预测值
        outputs = model_local_freeze(**batch)
        loss = outputs.loss          # 计算当前批次的损失
        logits = outputs.logits      # 获取模型的预测结果

        # 反向传播并更新参数
        optimizer.zero_grad()        # 清除优化器的梯度
        loss.backward()              # 计算梯度
        optimizer.step()             # 更新模型参数
        scheduler.step()             # 更新学习率

    # 记录训练结束时间
    end_time = time.time()
    train_time = end_time - start_time  # 计算训练时间
    print(f"Training time: {train_time:.2f} seconds")

    # 验证模型
    model_local_freeze.eval()  # 设置模型为评估模式

    # 记录验证开始时间
    start_time = time.time()

    for batch in val_loader:
        batch = {k: v.to(device) for k, v in zip(["input_ids", "attention_mask", "labels"], batch)}
        with torch.no_grad():  # 禁用梯度计算以节省内存和加速
            outputs = model_local_freeze(**batch)

        logits = outputs.logits                # 获取预测值
        predictions = torch.argmax(logits, dim=-1)  # 获取预测的类别
        metric.add_batch(predictions=predictions, references=batch["labels"])  # 更新评估指标

    # 记录验证结束时间
    end_time = time.time()
    val_time = end_time - start_time  # 计算验证时间
    print(f"Validation time: {val_time:.2f} seconds")

    # 计算并输出验证集的准确率
    val_accuracy = metric.compute()["accuracy"]
    print(f"Epoch {epoch + 1}: Validation Accuracy = {val_accuracy:.4f}")

## 3. BERTopic

In [ ]:
!pip install bertopic 
!pip install sentence-transformers 

In [ ]:
with open('data/FinancialPhraseBank/eng_sent.txt', 'r', encoding='utf-8') as file:
    content = file.read()
sentences = content.split('\n')
len(sentences)

In [ ]:
# #all-MiniLM-L6-v2 是一个轻量级 适用于BERTopic任务的嵌入模型，比之传统bert，快数倍
# #还是请夏威夷的同学下载一下
# from sentence_transformers import SentenceTransformer

# # 下载并保存模型
# model = SentenceTransformer("all-MiniLM-L6-v2")  # 下载模型
# model.save("./model/all-MiniLM-L6-v2")  # 保存模型到本地路径
# print("Model saved locally at ./models/all-MiniLM-L6-v2")

In [ ]:

# 加载本地模型
embedding_model = SentenceTransformer("./model/all-MiniLM-L6-v2")  # 从本地路径加载模型
print("Model loaded successfully from local path.")

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# 准备数据
embedding_model.to(device)
# 使用 SentenceTransformer 提取句子嵌入
topic_model = BERTopic(
    embedding_model=embedding_model,  # 使用自定义的 SentenceTransformer 模型生成嵌入
    top_n_words=5,                   # 每个主题显示的关键词数量，默认值为 10
    min_topic_size=20,               # 最小主题大小（即主题中至少包含的文档数量），默认值为 10
    nr_topics=30                   # 将主题数量限制为 30（可以是整数或 "auto"），默认值为 None
)

start_time = time.time()

# 使用 BERTopic 进行聚类
topics, probs = topic_model.fit_transform(sentences)

end_time = time.time()
dur_time = end_time - start_time  # 计算验证时间
print(f"Clustering time: {dur_time:.2f} seconds")


In [ ]:


# 查看主题信息
print("\nGenerated Topics:")
print(topic_model.get_topic_info())


In [ ]:
translated_lines[330]